In [ ]:
"""
A10 (Deca-Alanine) — Complete Setup, Equilibration & Production: Phase 1 → Phase 4
=====================================================================================

CORRECTIONS & ADDITIONS in this version
-----------------------------------------
[Phase 1] Capping:   Ac-Ala10-NH2 — ACE (acetyl) N-terminus + NME (N-methylamide)
                     C-terminus added via OpenMM Modeller after PeptideBuilder raw PDB.
                     Eliminates the +1/-1 charged terminus artifact that artificially
                     stabilizes the helix by ~1-3 kcal/mol for short peptides.
                     All literature (Kuczera 2021/2024, Zhuang 2021) uses capped termini.

[Phase 1] Temperature: 330 K (raised from 300 K). At 300 K the helix relaxation
                     time is ~20-30 ns; at 330 K kinetics are several-fold faster
                     (Arrhenius), giving more helix-coil transitions per ns of simulation.
                     This allows meaningful statistics from a 1 µs production run.

[Phase 2] FF:        charmm36_2024.xml + charmm36_2024/water.xml (CHARMM36m)
[Phase 2] Padding:   1.5 nm
[Phase 3] Restraints: positional restraints during minimization + NVT only
[Phase 3] Duration:  1 ns NVT + 10 ns NPT
[Phase 3] Re-solvation: After NPT of extended conformation, re-extract peptide,
                     re-solvate in a box matched to the helix box size, and run
                     a short 2 ns NPT to re-converge density. This follows
                     Kuczera et al. 2021 methodology exactly and ensures both
                     production systems have comparable atom counts and box sizes.

[Phase 4] Production: 1 µs NVT per conformation (barostat OFF; volume fixed from
                     NPT-equilibrated box). NVT chosen over NPT for production
                     because volume fluctuations add cost and noise without benefit
                     once density is converged (Kuczera 2021 rationale). Checkpoint/
                     resume support — safe to interrupt and continue.

Output files (per conformation):
  A10_{label}_minimized.pdb         — post-minimization snapshot
  A10_{label}_nvt.dcd               — Phase 3 NVT equilibration trajectory
  A10_{label}_nvt.log               — Phase 3 NVT log
  A10_{label}_nvt.chk               — Phase 3 NVT checkpoint
  A10_{label}_npt.dcd               — Phase 3 NPT equilibration trajectory
  A10_{label}_npt.log               — Phase 3 NPT log
  A10_{label}_npt.chk               — Phase 3 NPT checkpoint
  A10_{label}_npt_final.pdb         — Phase 3 final frame → Phase 4 input
  A10_{label}_npt_final.xml         — Phase 3 serialized system (no barostat)
  A10_{label}_prod.dcd              — Phase 4 production trajectory (1 µs)
  A10_{label}_prod.log              — Phase 4 production log
  A10_{label}_prod.chk              — Phase 4 checkpoint (resume support)
"""

# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import os
import sys
import time
import platform as _platform
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# WORKING DIRECTORY — ensure all relative file paths resolve to the notebook's
# own folder on every OS (Windows Jupyter already does this; Linux/RunPod may
# start Jupyter in /root or /workspace instead of the project directory).
# ─────────────────────────────────────────────────────────────────────────────
_nb_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()
os.chdir(_nb_dir)
print(f"Working directory: {os.getcwd()}")

try:
    from PeptideBuilder import Geometry
    import PeptideBuilder
    import Bio.PDB
except ImportError as _e:
    raise ImportError(
        "PeptideBuilder not found. Install with: pip install PeptideBuilder biopython"
    ) from _e

try:
    from pdbfixer import PDBFixer
    import openmm as mm
    import openmm.app as app
    from openmm import unit, XmlSerializer, Platform
except ImportError as _e:
    raise ImportError(
        "OpenMM or pdbfixer not found. Install with: conda install -c conda-forge openmm pdbfixer"
    ) from _e

try:
    import mdtraj as md
    MDTRAJ_AVAILABLE = True
except ImportError:
    print("WARNING: mdtraj not found. RMSD convergence check will be skipped.")
    print("         Install with: conda install -c conda-forge mdtraj")
    MDTRAJ_AVAILABLE = False

print("All dependencies loaded.\n")


# ─────────────────────────────────────────────────────────────────────────────
# WINDOWS-ONLY FIX: Patch OpenMM's safesave to avoid PermissionError on
# checkpoint writes. On Windows, atomic rename over an open file handle can
# fail; the patch retries with exponential back-off. On Linux os.replace() is
# already atomic so no patch is needed.
# ─────────────────────────────────────────────────────────────────────────────
import openmm.app.internal.safesave as _safesave
import tempfile as _tempfile

if _platform.system() == "Windows":
    def _patched_save(data, filename):
        """
        Drop-in replacement for openmm.app.internal.safesave.save that handles
        Windows PermissionError / FileExistsError on atomic rename by retrying.
        """
        import time as _time
        dir_ = os.path.dirname(os.path.abspath(filename))
        fd, tmp = _tempfile.mkstemp(dir=dir_, prefix=os.path.basename(filename) + ".tmp")
        try:
            os.write(fd, data if isinstance(data, (bytes, bytearray)) else data.encode())
        finally:
            os.close(fd)

        for attempt in range(6):
            try:
                os.replace(tmp, filename)
                return
            except (PermissionError, OSError):
                if attempt == 5:
                    try:
                        os.remove(tmp)
                    except OSError:
                        pass
                    raise
                _time.sleep(0.2 * (2 ** attempt))   # 0.2, 0.4, 0.8, 1.6, 3.2 s

    _safesave.save = _patched_save
    print("Windows detected -- applied checkpoint atomic-rename patch.\n")
else:
    print(f"Platform: {_platform.system()} -- native atomic rename used (no patch needed).\n")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1: BUILD PEPTIDE STRUCTURES WITH ACE/NME CAPPING
# ─────────────────────────────────────────────────────────────────────────────

def build_a10(phi: float, psi: float, output_path: str, label: str):
    """
    Build a capped 10-residue alanine peptide using CHARMM36m's native ACE/CT3 patch scheme.

    WHY CAPPING IS MANDATORY
    ------------------------
    Without caps the peptide has NH3+ (N-term, +1) and COO- (C-term, -1).
    These formal charges create an artificial macrodipole that stabilizes the
    alpha-helix by ~1-3 kcal/mol — inflating helix content relative to physical
    reality. Every published deca-alanine study (Kuczera 2021/2024, Zhuang 2021,
    NAMD tutorial) uses ACE/NME caps. This is non-negotiable for a correct
    free energy landscape.

    HOW CHARMM36m ACTUALLY IMPLEMENTS CAPS (critical insight)
    ----------------------------------------------------------
    Unlike AMBER force fields (which have standalone ACE/NME residue templates),
    CHARMM36m uses a PATCH mechanism:
      - ACE_0 patch: adds atoms CY, OY, CAY, HY1, HY2, HY3 to the N-terminal ALA
      - CT3_0 patch: adds atoms NT, HNT, CAT, HT1, HT2, HT3 to the C-terminal ALA
    The terminal residues remain named "ALA" but carry extra atoms. OpenMM's
    ForceField.createSystem() detects these extra atoms and automatically matches
    the patched template.

    WHY NOT WRITE ACE/NME AS SEPARATE RESIDUES
    -------------------------------------------
    CHARMM36m has NO standalone ACE or NME residue templates (verified by
    inspection of charmm36_2024.xml). Writing a PDB with residue names ACE or NME
    causes: "No template found for residue 0 (ACE)". The patch mechanism requires
    the cap atoms to be PART OF the terminal ALA residue records.

    WHY NOT USE app.PDBFile ROUNDTRIP FOR THE CAPPED STRUCTURE
    -----------------------------------------------------------
    OpenMM's PDB reader normalises hydrogen names: HT1->H, HT2->H2, HT3->H3,
    HN->H globally. This renames the CT3 methyl hydrogens (HT1/HT2/HT3 on CAT)
    to look like extra backbone amide protons, causing template mismatch.
    Solution: keep the correctly-named topology in-memory and never re-read
    it through app.PDBFile(). A PDB file is written only for visual inspection;
    parameterisation uses the in-memory topology directly.

    APPROACH (5 steps)
    ------------------
    1. PeptideBuilder builds 10 ALA residues (add_terminal_OXT for complete C-term).
    2. PDBFixer adds missing atoms and all hydrogens at pH 7.0.
       This gives NH3+/COO- termini — wrong, but gives us all backbone H atoms.
    3. Build a new OpenMM Topology, copying all PDBFixer atoms EXCEPT:
       - H2, H3 from first residue (extra NH3+ protons — conflict with ACE patch)
       - OXT, HXT from last residue (C-terminus carboxylate — conflict with CT3 patch)
       Also copy all bonds from PDBFixer for the kept atoms.
    4. Append ACE patch atoms (CY, OY, CAY, HY1, HY2, HY3) to the first ALA
       and CT3 patch atoms (NT, HNT, CAT, HT1, HT2, HT3) to the last ALA,
       with geometrically-placed coordinates and correct bonds.
    5. Write an inspection PDB (atom names preserved in this call because we
       use PDBFile.writeFile with the in-memory topology directly). Return
       the topology and positions for immediate use by prepare_system.

    Parameters
    ----------
    phi         : backbone phi dihedral (degrees) — applied to all residues
    psi         : backbone psi dihedral (degrees) — applied to all residues
    output_path : where to write the inspection PDB (for visual verification)
    label       : human-readable name used in log output

    Returns
    -------
    (topology, positions) — OpenMM Topology and Quantity[nanometer] positions
                            ready for app.Modeller() in prepare_system.
    """
    import openmm.app.element as elem_mod
    from openmm import Vec3

    raw_pdb = output_path.replace(".pdb", "_raw.pdb")

    # ── Step 1: PeptideBuilder — backbone heavy atoms, 10 ALA ────────────
    # add_terminal_OXT is called so PDBFixer sees a complete C-terminus and
    # assigns standard ALA residue templates. OXT will be removed in Step 3.
    geo         = Geometry.AlaGeo()
    geo.phi     = phi
    geo.psi_im1 = psi

    structure = PeptideBuilder.initialize_res(geo)
    for _ in range(9):
        PeptideBuilder.add_residue(structure, geo)
    PeptideBuilder.add_terminal_OXT(structure)  # needed for PDBFixer to add correct H

    io = Bio.PDB.PDBIO()
    io.set_structure(structure)
    io.save(raw_pdb)

    # ── Step 2: PDBFixer — complete all-atom structure ───────────────────
    # addMissingHydrogens(pH=7.0) adds H, H2, H3 to the N-terminus (NH3+)
    # and nothing extra to the C-terminus (already has OXT from Step 1).
    # We will strip the extra H2/H3 and OXT in Step 3.
    fixer = PDBFixer(filename=raw_pdb)
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(pH=7.0)

    src_top  = fixer.topology
    src_pos  = list(fixer.positions)
    res_list = list(src_top.residues())
    n_res    = len(res_list)

    # ── Step 3: Build new topology, removing conflicting terminal atoms ──
    # Atoms to remove from first residue (NH3+ extra protons from PDBFixer).
    # The ACE_0 patch keeps exactly ONE backbone amide H (= "H" after PDB read).
    # Atoms to remove from last residue (C-term carboxylate O from PDBFixer).
    del_first = {"H2", "H3"}
    del_last  = {"OXT", "HXT"}

    # Helper: get {name: coord_angstrom} for a residue
    def adict_of(res):
        return {a.name: np.array(src_pos[a.index].value_in_unit(unit.angstrom))
                for a in res.atoms()}

    af = adict_of(res_list[0])   # first ALA atom positions
    al = adict_of(res_list[-1])  # last  ALA atom positions

    # ── Step 4a: Compute ACE patch atom positions (Angstrom) ─────────────
    # ACE_0 patch adds to first ALA: CY (acetyl carbonyl C), OY (carbonyl O),
    # CAY (methyl C), HY1/HY2/HY3 (methyl H). Bond: CY-N (peptide bond).
    N_f  = af["N"];  CA_f = af["CA"];  O_f = af["O"];  C_f = af["C"]
    v_f  = (N_f - CA_f) / np.linalg.norm(N_f - CA_f)   # CA→N direction
    # Peptide-plane perpendicular for OY placement
    c1   = np.cross(v_f, O_f - C_f); c1 /= np.linalg.norm(c1)
    perp_f = np.cross(c1, v_f); perp_f /= np.linalg.norm(perp_f)
    CY   = N_f  + 1.33 * v_f         # peptide bond C-N = 1.33 Å
    OY   = CY   + 1.23 * perp_f      # C=O carbonyl
    CAY  = CY   + 1.52 * v_f         # C-CH3 = 1.52 Å
    r_h  = 1.09; a2 = np.radians(70.5)
    HY1  = CAY + r_h * np.array([ np.sin(a2), -np.cos(a2),  0.0])
    HY2  = CAY + r_h * np.array([-np.sin(a2)*0.5, -np.cos(a2),  np.sin(a2)*0.866])
    HY3  = CAY + r_h * np.array([-np.sin(a2)*0.5, -np.cos(a2), -np.sin(a2)*0.866])
    ace_extra = [
        ("CY",  elem_mod.carbon,    CY),
        ("OY",  elem_mod.oxygen,    OY),
        ("CAY", elem_mod.carbon,   CAY),
        ("HY1", elem_mod.hydrogen, HY1),
        ("HY2", elem_mod.hydrogen, HY2),
        ("HY3", elem_mod.hydrogen, HY3),
    ]

    # ── Step 4b: Compute CT3 patch atom positions (Angstrom) ─────────────
    # CT3_0 patch adds to last ALA: NT (methylamide N), HNT (amide H),
    # CAT (methyl C), HT1/HT2/HT3 (methyl H). Bond: C_last-NT (peptide bond).
    C_l  = al["C"];  CA_l = al["CA"]
    v_l  = (C_l - CA_l) / np.linalg.norm(C_l - CA_l)   # CA→C direction
    NT   = C_l  + 1.33 * v_l
    HNT  = NT   + np.array([0.0, 1.01, 0.0])
    CAT  = NT   + 1.47 * v_l
    HT1  = CAT  + r_h * np.array([ np.sin(a2), -np.cos(a2),  0.0])
    HT2  = CAT  + r_h * np.array([-np.sin(a2)*0.5, -np.cos(a2),  np.sin(a2)*0.866])
    HT3  = CAT  + r_h * np.array([-np.sin(a2)*0.5, -np.cos(a2), -np.sin(a2)*0.866])
    ct3_extra = [
        ("NT",  elem_mod.nitrogen,  NT),
        ("HNT", elem_mod.hydrogen, HNT),
        ("CAT", elem_mod.carbon,   CAT),
        ("HT1", elem_mod.hydrogen, HT1),
        ("HT2", elem_mod.hydrogen, HT2),
        ("HT3", elem_mod.hydrogen, HT3),
    ]

    # ── Build new topology with correct atoms ────────────────────────────
    new_top   = app.Topology()
    new_chain = new_top.addChain()
    new_pos   = []          # list of np.array([x,y,z]) in Angstrom
    old_to_new = {}         # old atom index -> new Atom object (for bond copying)
    ace_atoms  = {}         # name -> new Atom (for ACE bond wiring)
    ct3_atoms  = {}         # name -> new Atom (for CT3 bond wiring)

    for i, res in enumerate(res_list):
        skip  = (del_first if i == 0 else set()) | (del_last if i == n_res - 1 else set())
        new_res = new_top.addResidue(res.name, new_chain)

        # Copy kept atoms from PDBFixer
        for atom in res.atoms():
            if atom.name in skip:
                continue
            new_atom = new_top.addAtom(atom.name, atom.element, new_res)
            old_to_new[atom.index] = new_atom
            new_pos.append(np.array(src_pos[atom.index].value_in_unit(unit.angstrom)))

        # Append ACE patch atoms at end of first residue
        if i == 0:
            for aname, aelem, coord in ace_extra:
                new_atom = new_top.addAtom(aname, aelem, new_res)
                ace_atoms[aname] = new_atom
                new_pos.append(coord)

        # Append CT3 patch atoms at end of last residue
        if i == n_res - 1:
            for aname, aelem, coord in ct3_extra:
                new_atom = new_top.addAtom(aname, aelem, new_res)
                ct3_atoms[aname] = new_atom
                new_pos.append(coord)

    # Copy bonds between kept atoms from PDBFixer topology
    for bond in src_top.bonds():
        a1, a2_ = bond[0], bond[1]
        if a1.index in old_to_new and a2_.index in old_to_new:
            new_top.addBond(old_to_new[a1.index], old_to_new[a2_.index])

    # Wire ACE patch bonds
    def get_atom(res, name):
        return next(a for a in res.atoms() if a.name == name)

    first_res = list(new_top.residues())[0]
    last_res  = list(new_top.residues())[-1]
    N_first   = get_atom(first_res, "N")
    C_last_a  = get_atom(last_res,  "C")

    new_top.addBond(ace_atoms["CY"],  N_first)            # ACE carbonyl C – ALA N
    new_top.addBond(ace_atoms["CY"],  ace_atoms["OY"])    # C=O
    new_top.addBond(ace_atoms["CY"],  ace_atoms["CAY"])   # C-methyl
    new_top.addBond(ace_atoms["CAY"], ace_atoms["HY1"])
    new_top.addBond(ace_atoms["CAY"], ace_atoms["HY2"])
    new_top.addBond(ace_atoms["CAY"], ace_atoms["HY3"])

    # Wire CT3 patch bonds
    new_top.addBond(C_last_a,          ct3_atoms["NT"])   # ALA C – methylamide N
    new_top.addBond(ct3_atoms["NT"],   ct3_atoms["HNT"])
    new_top.addBond(ct3_atoms["NT"],   ct3_atoms["CAT"])
    new_top.addBond(ct3_atoms["CAT"],  ct3_atoms["HT1"])
    new_top.addBond(ct3_atoms["CAT"],  ct3_atoms["HT2"])
    new_top.addBond(ct3_atoms["CAT"],  ct3_atoms["HT3"])

    # ── Step 5: Convert to OpenMM positions quantity and write PDB ───────
    positions_nm = [Vec3(*(p / 10.0)) for p in new_pos] * unit.nanometer

    # Write an inspection PDB. NOTE: app.PDBFile.writeFile preserves the exact
    # atom names from the in-memory topology (unlike app.PDBFile() *reading*,
    # which renames HT1→H, HNT→H, etc.). The written PDB is for visual
    # inspection only; parameterisation uses the in-memory topology directly.
    with open(output_path, "w") as f:
        app.PDBFile.writeFile(new_top, positions_nm, f)

    res_names = [r.name for r in new_top.residues()]
    atoms_all = list(new_top.atoms())
    first_r   = list(new_top.residues())[0]
    last_r    = list(new_top.residues())[-1]
    has_ace   = any(a.name in {"CY", "OY", "CAY"} for a in first_r.atoms())
    has_ct3   = any(a.name in {"NT", "CAT"}        for a in last_r.atoms())

    print(f"  [{label}] -> {output_path}")
    print(f"    Residues : {len(res_names)}  ({', '.join(res_names)})")
    print(f"    Atoms    : {len(atoms_all)}  (all-atom including H)")
    print(f"    phi/psi  : {phi:.1f} / {psi:.1f} deg")
    if has_ace and has_ct3:
        print("    Capping  : OK ACE patch (CY/OY/CAY on first ALA) + CT3 patch (NT/CAT on last ALA)")
        print("               Ac-Ala10-NHMe -- neutral termini as required by CHARMM36m")
    else:
        print(f"    WARN CAPPING CHECK FAILED: has_ace={has_ace}  has_ct3={has_ct3}")
    print()

    return new_top, positions_nm


print("=" * 60)
print("  PHASE 1 -- Build A10 Structures (ACE/NME capped)")
print("=" * 60)

# Extended β-strand: φ = −120°, ψ = 140°
# This is the fully extended PPII/β-strand conformation.
# φ = -120° (well into the β region of the Ramachandran map)
# ψ = +140° (extended, not folded)
# Starting from extended ensures we are NOT biasing towards the helix
# and can observe folding events during production.
top_ext_pep, pos_ext_pep = build_a10(
    phi=-120.0, psi=140.0, output_path="A10_extended.pdb", label="Extended"
)

# Alpha-helix: φ = −57°, ψ = −47°
# These are the canonical α-helix Ramachandran angles.
# Starting from helix ensures we observe unfolding events.
# Running both conformations and checking that thermodynamic quantities
# (helix content, free energy) converge from both directions is a standard
# convergence test for helix-coil simulations.
top_hel_pep, pos_hel_pep = build_a10(
    phi=-57.0, psi=-47.0, output_path="A10_helix.pdb", label="Alpha-Helix"
)

print("[Phase 1] OK Both capped peptide topologies built (Ac-Ala10-NHMe via CHARMM ACE/CT3 patches).\n")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2: PARAMETERIZE & SOLVATE
# ─────────────────────────────────────────────────────────────────────────────

def prepare_system(peptide_topology, peptide_positions,
                   solvated_pdb: str, xml_out: str, label: str,
                   padding_nm: float = 1.5):
    """
    Solvate and parameterize a capped A10 conformation.

    FORCE FIELD: charmm36_2024.xml (CHARMM36m)
    -------------------------------------------
    CHARMM36m (MacKerell 2017) is the gold-standard protein force field for
    helix-coil equilibria in small peptides. The critical difference from
    CHARMM36 is the CMAP correction to backbone phi/psi torsions, which was
    re-fitted against NMR J-coupling data and quantum chemical torsion scans
    specifically to improve backbone conformational preferences in intrinsically
    disordered peptides and helix-coil systems. For deca-alanine, CHARMM36m
    gives ~15-20% helix content, consistent with CD spectroscopy (Kuczera 2021).
    AMBER ff14SB is also valid but CHARMM36m is preferred when using TIP3P water
    because they were co-optimized.

    WATER MODEL: TIP3P (charmm36_2024/water.xml)
    ---------------------------------------------
    TIP3P is the standard water model for CHARMM36m. It is a rigid 3-point model
    parameterized at 300 K for bulk water. Its known limitation is that bulk water
    diffusion is too fast (~2.3×10-9 m²/s vs experimental ~2.3×10-9 m²/s at 298 K —
    actually reasonable here), but for peptide-in-water simulations TIP3P is
    appropriate because the protein force field was parameterized against TIP3P.
    Using TIP4P-Ew or TIP5P with CHARMM36m would introduce systematic errors.
    We use charmm36_2024/water.xml (standard CHARMM TIP3P), NOT tip3p-pme-b.xml
    (which has modified charges not co-optimized with CHARMM36m backbone parameters).

    PADDING: 1.5 nm
    ---------------
    The periodic box edge is the peptide extent + 2×padding.
    Extended A10 is ~3.5 nm end-to-end. At 1.2 nm padding the box edge would be
    ~6.2 nm and the minimum image distance from peptide to its periodic copy would
    be only ~2.7 nm — shorter than twice the 1.2 nm cutoff (2.4 nm). The peptide
    would interact with its own ghost image. At 1.5 nm padding the minimum image
    distance is ~3.0 nm > 2×cutoff = 2.4 nm. This is the correct criterion.
    For the helix (~1.5 nm end-to-end) 1.5 nm padding is comfortably conservative.

    IONIC STRENGTH: 0.15 M NaCl
    ----------------------------
    Physiological ionic strength. NaCl ions serve two purposes:
    1. Screen long-range electrostatic interactions to mimic cytoplasm.
    2. Ensure charge neutrality — without neutralize=True, the net charge of
       the system would give a non-zero total charge, causing a divergent PME
       sum and unphysical results.
    0.15 M is the standard used in all benchmark studies (Kuczera 2021, NAMD tutorial).

    NONBONDED METHOD: PME (Particle Mesh Ewald)
    --------------------------------------------
    PME is mandatory for any periodic solvated system with charged atoms or partial
    charges. Direct truncation of Coulomb forces at a cutoff introduces large
    systematic errors in potential energy (~10-100 kJ/mol for polar systems). PME
    computes long-range electrostatics exactly (to machine precision) using FFT
    interpolation on a reciprocal-space grid. ewaldErrorTolerance=0.0005 sets the
    relative RMS error in forces, which determines the grid spacing. 0.0005 is the
    CHARMM36m recommended value (Kim et al. 2015, J Chem Theory Comput).

    NONBONDED CUTOFF: 1.2 nm
    -------------------------
    CHARMM36m was parameterized at 1.2 nm real-space cutoff. Using a shorter cutoff
    (e.g. 1.0 nm) systematically underestimates VDW attraction in the 1.0-1.2 nm
    range, producing ~5-10% too-high densities and incorrect helix stability.
    Using a longer cutoff (>1.4 nm) has negligible accuracy benefit but increases
    neighbor list size and slows computation by ~20-40%.

    SWITCH DISTANCE: 1.0 nm (force-switching)
    -----------------------------------------
    CHARMM force fields use a force-switch function (not an energy-switch or hard
    truncation) that smoothly attenuates the VDW force between 1.0 and 1.2 nm to
    zero at the cutoff. This avoids the large force discontinuity at the cutoff
    that causes energy drift. Without the switch, VDW forces jump discontinuously
    at 1.2 nm → artificial heating/cooling artifacts in long simulations.
    switchDistance=1.0 nm means: VDW is unmodified for r < 1.0 nm, smoothly
    switched to zero for 1.0 < r < 1.2 nm.

    CONSTRAINTS: HBonds
    -------------------
    HBonds constrains all bonds involving hydrogen (X-H bonds) to their equilibrium
    length. This removes the fastest vibrational modes (3000-3500 cm-1 O-H stretch,
    2800-3000 cm-1 C-H stretch), which is what limits the timestep to ~1 fs without
    constraints. With HBonds constraints, a 2 fs timestep is safe. Using AllBonds
    would also constrain heavy-atom bonds (C-C, C-N, C-O), which would require
    verification that the constraint solver converges for stiff bonds — HBonds is
    the safer and more standard choice.

    RIGID WATER: True
    -----------------
    TIP3P is a rigid water model by construction. rigidWater=True tells OpenMM to
    apply SETTLE (an analytical rigid body integrator) to water molecules instead
    of constraint iteration. SETTLE is exact for 3-atom rigid bodies and much
    faster than iterative SHAKE/LINCS for the thousands of water molecules here.
    This is always correct for TIP3P — there is no reason to set it False.

    CONSTRAINTS INTERACTION WITH TIMESTEP
    -------------------------------------
    With HBonds + rigidWater, the remaining slowest modes are heavy-atom bending
    vibrations (~300-500 cm-1). These limit the safe timestep to ~4 fs with
    HMR (hydrogen mass repartitioning) or ~2 fs without. We use 2 fs, which is
    universally safe and does not require HMR.
    """
    print(f"  Processing [{label}]")

    # charmm36_2024.xml = protein + ions parameters with CHARMM36m torsion CMAP
    # charmm36_2024/water.xml = TIP3P water co-parameterized with CHARMM36m
    forcefield = app.ForceField("charmm36_2024.xml", "charmm36_2024/water.xml")
    # Use topology/positions directly from build_a10 — do NOT re-read the PDB
    # through app.PDBFile(), which would rename HT1→H, HT2→H2, etc., destroying
    # the CHARMM patch atom names required by charmm36_2024.xml.
    modeller   = app.Modeller(peptide_topology, peptide_positions)

    modeller.addSolvent(
        forcefield,
        model         = "tip3p",          # rigid 3-point; co-parameterized with CHARMM36m
        padding       = padding_nm * unit.nanometers,
        ionicStrength = 0.15 * unit.molar, # physiological; screens electrostatics
        neutralize    = True,              # mandatory — PME diverges for net-charged systems
    )

    n_atoms  = modeller.topology.getNumAtoms()
    n_waters = sum(1 for r in modeller.topology.residues() if r.name == "HOH")
    n_na     = sum(1 for r in modeller.topology.residues() if r.name == "NA")
    n_cl     = sum(1 for r in modeller.topology.residues() if r.name == "CL")
    print(f"    Padding  : {padding_nm} nm")
    print(f"    Atoms    : {n_atoms:,}")
    print(f"    Waters   : {n_waters:,}  |  Na+: {n_na}  |  Cl-: {n_cl}")

    with open(solvated_pdb, "w") as f:
        app.PDBFile.writeFile(modeller.topology, modeller.positions, f)
    print(f"    Saved    : {solvated_pdb}")

    system = forcefield.createSystem(
        modeller.topology,
        nonbondedMethod     = app.PME,                    # exact long-range electrostatics
        nonbondedCutoff     = 1.2 * unit.nanometers,      # CHARMM36m parameterization cutoff
        switchDistance      = 1.0 * unit.nanometers,      # force-switch onset: CHARMM convention
        constraints         = app.HBonds,                 # freeze X-H bonds → 2 fs timestep safe
        rigidWater          = True,                       # SETTLE for TIP3P — exact + fast
        ewaldErrorTolerance = 0.0005,                     # PME grid density; CHARMM36m default
    )

    n_constr = system.getNumConstraints()
    # Correct DOF = 3N - N_constraints - 3
    # The -3 accounts for CMMotionRemover which removes 3 translational DOF.
    # Using naive 3N overestimates DOF by ~33% → instantaneous T computed from
    # KE = (DOF/2)*kB*T would be wrong by ~33%. This matters for convergence checks.
    dof = 3 * n_atoms - n_constr - 3
    print(f"    DOF      : {dof:,}   (naive 3N = {3*n_atoms:,})")
    print(f"    Forces   : {[type(f).__name__ for f in system.getForces()]}")

    # Serialize to XML so Phase 3 can deserialize fresh copies.
    # Serialization + deserialization is the only safe way to share a System
    # between NVT and NPT — Python objects share memory; modifying one
    # (e.g. adding a barostat) would corrupt the other.
    with open(xml_out, "w") as f:
        f.write(XmlSerializer.serialize(system))
    print(f"    Saved    : {xml_out}\n")

    return system, modeller.topology, modeller.positions, dof


print("=" * 60)
print("  PHASE 2 -- Parameterize & Solvate")
print("=" * 60)
print("  Force field : charmm36_2024.xml (CHARMM36m backbone torsions)")
print("  Water model : charmm36_2024/water.xml (CHARMM TIP3P)")
print("  Capping     : CHARMM ACE patch (first ALA) + CT3 patch (last ALA) -- Ac-Ala10-NHMe\n")

system_ext, top_ext, pos_ext, dof_ext = prepare_system(
    peptide_topology=top_ext_pep,    peptide_positions=pos_ext_pep,
    solvated_pdb="A10_extended_solvated.pdb",
    xml_out="A10_extended.xml",      label="Extended",
)
system_hel, top_hel, pos_hel, dof_hel = prepare_system(
    peptide_topology=top_hel_pep,    peptide_positions=pos_hel_pep,
    solvated_pdb="A10_helix_solvated.pdb",
    xml_out="A10_helix.xml",         label="Alpha-Helix",
)
print("[Phase 2] OK Both systems parameterized and serialized.\n")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3: ENERGY MINIMIZATION + NVT + NPT EQUILIBRATION
# ─────────────────────────────────────────────────────────────────────────────

# ── Configuration ─────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────
# SIMULATION PARAMETERS — SCIENTIFIC RATIONALE FOR EVERY CHOICE
# ─────────────────────────────────────────────────────────────────────────────

TIMESTEP_FS         = 2.0
# WHY 2 fs:
# The Nyquist criterion for numerical integration: the timestep must be at most
# 1/10 of the fastest vibrational period. The fastest mode with HBonds+rigidWater
# is heavy-atom bending (~400 cm-1 → period ~83 fs). So the safe limit is ~8 fs.
# We use 2 fs as the practical standard in CHARMM36m simulations — conservative
# enough to avoid energy drift even for stiff bonds near the constraint cutoff.
# 4 fs is achievable with HMR (hydrogen mass repartitioning) but HMR changes
# effective masses and can subtly affect dynamics; 2 fs avoids that complexity.
# Literature: every benchmark paper (Kuczera, Kim 2015, NAMD tutorial) uses 2 fs.

TEMPERATURE_K       = 330.0
# WHY 330 K (not 300 K, not 340 K):
# The helix-coil transition in deca-alanine has an activation energy of
# ~8-12 kcal/mol. The Arrhenius rate ratio k(T2)/k(T1) = exp[Ea/R × (1/T1 - 1/T2)].
# At Ea=10 kcal/mol: k(330K)/k(300K) ≈ 3-5×. This gives τ_fold at 330 K ≈ 40-115 ns
# (down from ~200-350 ns at 300 K), allowing 3-11 complete helix↔coil cycles in
# a 500 ns simulation — sufficient for a reliable FES and two-state characterisation.
# WHY NOT 320 K: only ~2-3 cycles in 500 ns at 320 K — too few.
# WHY NOT 340 K: helix content drops to <5% at 340 K; helix basin too sparse.
# 330 K provides ~8-15% helix fraction for A10 and adequate kinetic sampling.
# Reference: Kuczera et al. Life 2021 (300 K kinetics), Arrhenius-extrapolated.

PRESSURE_BAR        = 1.0
# WHY 1 bar:
# Standard atmospheric pressure. 1 bar = 100 kPa = 0.987 atm.
# For solution-phase simulations of small peptides, pressure has negligible
# effect on conformational equilibria (ΔV for helix↔coil is <5 mL/mol →
# pressure effect on ΔG is <0.01 kcal/mol at 1 bar). We use bar (SI) not atm.
# This only matters during NPT equilibration; Phase 4 production is NVT.

FRICTION_PS         = 1.0
# WHY 1 ps⁻¹ (Langevin friction coefficient γ):
# The Langevin thermostat combines Newtonian dynamics with a stochastic friction
# term that drives the system to the Boltzmann distribution at TEMPERATURE_K.
# γ = 1 ps⁻¹ is the recommended value for protein-in-water from CHARMM36m
# developers. At γ=1 ps⁻¹ the thermostat coupling time is ~1 ps, which is:
#   - Fast enough to correct temperature fluctuations from force errors
#   - Slow enough not to overdamp backbone dynamics
# γ too high (>10 ps⁻¹): overdamps protein motions → artificially slow folding
# γ too low (<0.1 ps⁻¹): poor temperature control → T drifts during long runs
# Reference: Goga et al. J Chem Theory Comput 2012, CHARMM36m paper SI.

NVT_NS              = 1.0
# WHY 1 ns NVT:
# The NVT phase has positional restraints on the peptide; only water is free.
# The purpose is to let water molecules diffuse around the peptide surface and
# fill cavities before density equilibration (NPT). TIP3P water at 330 K has
# a self-diffusion coefficient of ~7×10-9 m²/s. The time for a water molecule
# to diffuse one peptide diameter (~1.5 nm) is t = r²/6D ≈ 0.05 ns.
# 1 ns provides ~20× that timescale, ensuring complete water shell equilibration
# before releasing the restraints in NPT. Literature minimum is 0.5 ns; we use
# 1 ns for safety.

NPT_NS              = 10.0
# WHY 10 ns NPT:
# After restraints are released, both peptide and water equilibrate together.
# The box density converges on the timescale of water diffusion across the box
# (~1-2 ns for these box sizes) plus barostat relaxation (~0.5-1 ns per
# Monte Carlo barostat cycle). The peptide also begins to relax from its
# idealized starting geometry.
# 10 ns is long enough to:
#   1. Converge density to within ±5 kg/m³ of 1000 kg/m³
#   2. Partially equilibrate the peptide backbone at the interface
# The standard in benchmark studies is 5-20 ns NPT. We use 10 ns as the
# validated minimum from Kim et al. 2015 and Kuczera 2021.
# KNOWN LIMITATION: 10 ns NPT is sufficient for density convergence but NOT
# for conformational equilibration of the peptide itself. That is the purpose
# of the 500 ns production run, not the equilibration.

NPT_RESOLVATE_NS    = 2.0
# WHY 2 ns for re-solvation NPT:
# After the extended conformation is re-solvated in a smaller box, the new
# water shell needs to equilibrate around the (now partially-collapsed) peptide.
# 2 ns is sufficient because the box is small and the peptide is already
# partially equilibrated from the preceding 10 ns NPT.

PROD_US             = 0.5   # 500 ns per replica
# WHY 500 ns (not 1 µs):
# At 330 K τ_fold ≈ 40-115 ns, τ_unfold ≈ 5-27 ns. Expected complete helix↔coil
# cycles in 500 ns: T / (τ_fold + τ_unfold) ≈ 500 / (45-142) ≈ 3-11 cycles.
# This is the minimum for characterising FES basin topology and providing a
# ground-truth reference for the seeding experiment. Convergence is verified
# by comparing helix-start vs extended-start FES (two-replica agreement).
# 1 µs would be ideal but is impractical on a laptop RTX 4050 Ti where
# 500 ns × 2 replicas already requires ~4-6 days of continuous GPU compute.
# KNOWN LIMITATION: 3-11 cycles is sufficient for FES topology and f_helix
# to ±5%, but not for high-precision kinetics (need ≥50 cycles for rates
# to ±10%). Kinetic estimates at this length should be treated as indicative.

LOG_INTERVAL_PS     = 10.0
# WHY 10 ps log interval:
# Logs T, PE, KE every 10 ps. This gives 100,000 data points per µs —
# enough to compute mean temperature, energy fluctuations, and detect any
# slow drift. Logging more frequently (1 ps) would produce a 10× larger log
# file without additional insight, and the I/O overhead is measurable on some
# systems.

DCD_INTERVAL_PS     = 100.0
# WHY 100 ps DCD interval (not 500 ps):
# The Nyquist criterion for trajectory analysis: need frames at most every
# τ_relax/10 to resolve the autocorrelation function. With τ_relax ~5 ns,
# 100 ps gives 50 frames per relaxation time — more than sufficient.
# Storage: at 100 ps intervals, 500 ns / 100 ps = 5,000 frames per replica.
# At ~48 KB/frame for ~5,000 atoms: ~240 MB per DCD. Manageable on a laptop.
# WHY NOT 500 ps: 500 ps gives only 1,000 frames per replica (10 frames per τ_relax).
# This is the absolute minimum for TICA covariance estimation and reduces
# diagnostic resolution. 100 ps is preferred for safety — finer sampling
# catches artefacts and gives more robust TICA lag-scan data.
# KNOWN LIMITATION: Sub-ns backbone motions are not resolved at 100 ps, but
# these are not needed for helix-coil (binary state) analysis.

CHK_INTERVAL_PS     = 1000.0
# WHY 1 ns checkpoint interval:
# Checkpoint files contain the complete simulation state (positions, velocities,
# box vectors, RNG state). On a crash or kernel interrupt, we lose at most 1 ns
# of trajectory. More frequent checkpointing (100 ps) wastes I/O for marginal
# benefit; less frequent (10 ns) means losing ~10 ns on crash.

MIN_MAX_ITER        = 5000
# WHY 5000 minimization steps:
# L-BFGS minimization converges in 100-500 iterations for a well-behaved system.
# 5000 is a generous upper bound — if minimization has not converged in 5000 steps,
# the system likely has a major clash that minimization alone cannot fix.
# The tolerance check (below) will catch non-convergence.

MIN_TOLERANCE_KJ    = 10.0
# WHY 10 kJ/mol/nm tolerance:
# This is the maximum force component allowed at the end of minimization.
# 10 kJ/mol/nm = 1 kcal/mol/Å, a standard CHARMM protocol value.
# Tighter tolerance (1 kJ/mol/nm) is sometimes used but rarely necessary —
# the subsequent NVT equilibration will relax any remaining strain. Too tight
# a tolerance wastes compute time on diminishing returns.

RESTRAINT_K         = 1000.0
# WHY 1000 kJ/mol/nm² restraints on peptide heavy atoms:
# During NVT, we want water to equilibrate around the peptide, NOT the peptide
# to change conformation (which would negate the point of starting from two
# distinct conformations). 1000 kJ/mol/nm² means a 0.1 nm (1 Å) displacement
# of a restrained atom costs 10 kJ/mol — larger than typical thermal energy
# (~2.5 kJ/mol at 330 K) so the peptide is effectively locked.
# Too weak (10 kJ/mol/nm²): peptide drifts during NVT → mixed starting state
# Too strong (100,000 kJ/mol/nm²): rigid body, water can't relax to natural
# solvation geometry around flexible peptide geometry. 1000 is the CHARMM
# standard and validated in Kim et al. 2015.
# IMPORTANT: Restraints are ONLY active during minimization + NVT.
# NPT uses a freshly deserialized system with NO restraint force.

# ── Derived step counts (not parameters — computed from above) ────────────────
STEPS_PER_PS        = int(1000 / TIMESTEP_FS)        # = 500 steps/ps at 2 fs
NVT_STEPS           = int(NVT_NS            * 1000  * STEPS_PER_PS)
NPT_STEPS           = int(NPT_NS            * 1000  * STEPS_PER_PS)
NPT_RESOLVATE_STEPS = int(NPT_RESOLVATE_NS  * 1000  * STEPS_PER_PS)
PROD_STEPS          = int(PROD_US           * 1_000_000 * STEPS_PER_PS)
LOG_STEPS           = int(LOG_INTERVAL_PS   * STEPS_PER_PS)
DCD_STEPS           = int(DCD_INTERVAL_PS   * STEPS_PER_PS)
CHK_STEPS           = int(CHK_INTERVAL_PS   * STEPS_PER_PS)

CONFORMATIONS = [
    {
        "label":   "extended",
        "pdb":     "A10_extended_solvated.pdb",
        "xml":     "A10_extended.xml",
        "pep_top": top_ext_pep,   # in-memory topology with correct CHARMM atom names
        "pep_pos": pos_ext_pep,   # corresponding positions (nanometers)
    },
    {
        "label":   "helix",
        "pdb":     "A10_helix_solvated.pdb",
        "xml":     "A10_helix.xml",
        "pep_top": top_hel_pep,
        "pep_pos": pos_hel_pep,
    },
]


# ── Helpers ────────────────────────────────────────────────────────────────────

def get_best_platform():
    """Returns fastest available OpenMM Platform: CUDA > OpenCL > CPU > Reference.

    For CUDA, sets mixed precision and device 0 explicitly so the choice is
    reproducible across OpenMM builds (some default to 'single').
    """
    preference = ["CUDA", "OpenCL", "CPU", "Reference"]
    available  = {Platform.getPlatform(i).getName(): Platform.getPlatform(i)
                  for i in range(Platform.getNumPlatforms())}
    for name in preference:
        if name in available:
            plat = available[name]
            print(f"    Platform : {name}")
            if name == "CUDA":
                plat.setPropertyDefaultValue("CudaPrecision", "mixed")
                plat.setPropertyDefaultValue("CudaDeviceIndex", "0")
                print("    CudaPrecision  : mixed")
                print("    CudaDeviceIndex: 0")
            elif name == "CPU":
                try:
                    threads = plat.getPropertyDefaultValue("CpuThreads")
                    print(f"    Threads  : {threads}")
                except Exception:
                    pass
            return plat
    return Platform.getPlatform(0)


def compute_dof(system: mm.System) -> int:
    """
    Correct degrees of freedom for a solvated system:
      DOF = 3N - N_constraints - 3
    The final -3 comes from CMMotionRemover (added automatically by OpenMM
    when using Langevin integrators). Using the naive 3N overestimates by
    ~33% for a typical HBonds + rigidWater solvated system, giving a
    systematically wrong instantaneous temperature when computed from KE.
    """
    return 3 * system.getNumParticles() - system.getNumConstraints() - 3


def add_positional_restraints(system: mm.System, topology, positions, k: float) -> int:
    """
    Add a harmonic positional restraint force to all peptide heavy atoms.

    Uses periodicdistance() which correctly handles PBC wrapping — if a
    restrained atom crosses the periodic boundary, the force is computed
    from the minimum-image distance to the reference position rather than
    the absolute coordinate difference, preventing force blow-ups.

    Atoms restrained: all non-water, non-ion, non-hydrogen atoms (the peptide backbone
    and Cβ atoms — 51 atoms for A10 with OXT at the C-terminus).

    Parameters
    ----------
    system    : the OpenMM System to add the force to (modified in-place)
    topology  : OpenMM Topology matching the system
    positions : reference positions (from PDB) to restrain towards
    k         : force constant in kJ/mol/nm²

    Returns
    -------
    int : number of atoms that received restraints
    """
    restraint = mm.CustomExternalForce(
        "k * periodicdistance(x, y, z, x0, y0, z0)^2"
    )
    restraint.addGlobalParameter("k", k * unit.kilojoules_per_mole / unit.nanometers**2)
    restraint.addPerParticleParameter("x0")
    restraint.addPerParticleParameter("y0")
    restraint.addPerParticleParameter("z0")

    n_restrained = 0
    for atom in topology.atoms():
        if (atom.residue.name not in ("HOH", "NA", "CL")
                and atom.element is not None
                and atom.element.symbol != "H"):
            pos = positions[atom.index]
            restraint.addParticle(atom.index, [
                pos.x, pos.y, pos.z
            ])
            n_restrained += 1

    system.addForce(restraint)
    return n_restrained


def compute_density_kgm3(system: mm.System, state) -> float:
    """Density in kg/m3 from total system mass and periodic box volume.

    Uses the general triclinic-box formula V = a . (b x c) so it is correct
    for both orthogonal and non-orthogonal (triclinic) boxes.
    """
    total_mass_kg = sum(
        system.getParticleMass(i).value_in_unit(unit.dalton)
        for i in range(system.getNumParticles())
    ) * 1.66054e-27
    box = state.getPeriodicBoxVectors()
    a = np.array([v.value_in_unit(unit.meter) for v in box[0]])
    b = np.array([v.value_in_unit(unit.meter) for v in box[1]])
    c = np.array([v.value_in_unit(unit.meter) for v in box[2]])
    vol = abs(np.dot(a, np.cross(b, c)))
    return total_mass_kg / vol


def section(title: str):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")


# ── Main equilibration function ─────────────────────────────────────────────

def run_equilibration(conf: dict):
    label    = conf["label"]
    pdb_path = conf["pdb"]
    xml_path = conf["xml"]
    # Peptide topology with correct CHARMM atom names (HT1/HT2/HT3 preserved).
    # Used for re-solvation to avoid the app.PDBFile roundtrip that renames
    # HT1→H, HT2→H2, HT3→H3, causing template mismatch in createSystem.
    peptide_top_orig = conf.get("pep_top")
    peptide_pos_orig = conf.get("pep_pos")

    section(f"A10 [{label.upper()}] — Phase 3 Equilibration")
    print(f"  Input PDB : {pdb_path}")
    print(f"  Input XML : {xml_path}")
    print(f"  NVT       : {NVT_NS} ns  ({NVT_STEPS:,} steps)")
    print(f"  NPT       : {NPT_NS} ns  ({NPT_STEPS:,} steps)")
    print(f"  Timestep  : {TIMESTEP_FS} fs")

    for path in [pdb_path, xml_path]:
        if not os.path.exists(path):
            print(f"\nERROR: {path} not found. Run Phase 2 first.")
            return

    # ── Load structure and system ────────────────────────────────────────
    pdb = app.PDBFile(pdb_path)
    with open(xml_path) as f:
        system_nvt = XmlSerializer.deserialize(f.read())

    N   = system_nvt.getNumParticles()
    DOF = compute_dof(system_nvt)
    kB  = 8.314e-3  # kJ/mol/K

    print(f"\n  Atoms      : {N:,}")
    print(f"  DOF        : {DOF:,}  (naive 3N = {3*N:,}, difference = {3*N - DOF:,})")

    platform = get_best_platform()

    # ─────────────────────────────────────────────────────────────────────
    # STEP 1 — ENERGY MINIMIZATION (with positional restraints on peptide)
    # ─────────────────────────────────────────────────────────────────────
    section(f"[{label}] Step 1: Energy Minimization")

    # Add restraints to system_nvt BEFORE building any Simulation.
    # IMPORTANT: these restraints persist through NVT as well.
    # They are NOT present in system_npt (fresh deserialization below).
    n_restrained = add_positional_restraints(system_nvt, pdb.topology, pdb.positions,
                                             RESTRAINT_K)
    print(f"  Restraints applied to {n_restrained} heavy atoms (k = {RESTRAINT_K} kJ/mol/nm2)")
    print(f"  Restraints will persist through NVT, then released for NPT.\n")

    # INTEGRATOR CHOICE: LangevinMiddleIntegrator
    # -------------------------------------------
    # LangevinMiddleIntegrator uses the BAOAB splitting (B=force, A=free flow,
    # O=Ornstein-Uhlenbeck/thermostat). The 'Middle' refers to applying the
    # thermostat in the middle of the velocity update step, which gives better
    # configurational sampling than the older Langevin integrator (which applied
    # friction at the end). Specifically, LangevinMiddleIntegrator gives more
    # accurate average potential energies and structural properties at 2 fs
    # compared to the older LangevinIntegrator.
    # Reference: Zhang et al. J Chem Phys 2019 — introduced BAOAB / Middle scheme.
    #
    # WHY NOT Verlet + separate thermostat:
    # OpenMM's Verlet integrator does not include a thermostat; you would need to
    # add an AndersenThermostat Force. The Andersen thermostat uses random velocity
    # reassignment (not friction) — this breaks time-correlation functions and
    # makes autocorrelation-based relaxation time analysis invalid. Langevin
    # dynamics with friction preserves dynamical continuity and is the correct
    # choice for computing relaxation times.
    #
    # WHY NOT Nosé-Hoover:
    # Nosé-Hoover (via OpenMM's NoseHooverIntegrator) is correct for NPT
    # with a chain thermostat but is not implemented for NVT in OpenMM's
    # standard integrators and has known issues with the extended Hamiltonian
    # in OpenMM. Langevin at γ=1 ps-1 is the standard validated choice.
    integrator_min = mm.LangevinMiddleIntegrator(
        TEMPERATURE_K * unit.kelvin,
        FRICTION_PS   / unit.picosecond,
        TIMESTEP_FS   * unit.femtoseconds,
    )
    sim_min = app.Simulation(pdb.topology, system_nvt, integrator_min, platform)
    sim_min.context.setPositions(pdb.positions)

    state_pre = sim_min.context.getState(getEnergy=True)
    pe_pre    = state_pre.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
    print(f"  Pre-minimization PE  : {pe_pre:>12.1f} kJ/mol")

    t0 = time.time()
    sim_min.minimizeEnergy(
        tolerance     = MIN_TOLERANCE_KJ * unit.kilojoules_per_mole / unit.nanometer,
        maxIterations = MIN_MAX_ITER,
    )
    t_min = time.time() - t0

    state_post = sim_min.context.getState(getEnergy=True, getPositions=True)
    pe_post    = state_post.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
    print(f"  Post-minimization PE : {pe_post:>12.1f} kJ/mol")
    print(f"  Energy drop          : {pe_pre - pe_post:>12.1f} kJ/mol")
    print(f"  Wall time            : {t_min:.1f} s")

    min_pdb = f"A10_{label}_minimized.pdb"
    with open(min_pdb, "w") as f:
        app.PDBFile.writeFile(pdb.topology, state_post.getPositions(), f)
    print(f"  Saved                : {min_pdb}")

    # ─────────────────────────────────────────────────────────────────────
    # STEP 2 — NVT EQUILIBRATION (restraints still active from Step 1)
    # Supports resumption: if a .chk exists, load it and run only the
    # remaining steps instead of starting NVT from scratch.
    # ─────────────────────────────────────────────────────────────────────
    section(f"[{label}] Step 2: NVT — {NVT_NS} ns at {TEMPERATURE_K} K")
    print(f"  Peptide positional restraints remain active during NVT.")
    print(f"  Water relaxes freely around the restrained backbone.\n")

    integrator_nvt = mm.LangevinMiddleIntegrator(
        TEMPERATURE_K * unit.kelvin,
        FRICTION_PS   / unit.picosecond,
        TIMESTEP_FS   * unit.femtoseconds,
    )
    # setRandomNumberSeed(42): fixes the stochastic noise in the Langevin thermostat
    # for reproducibility. The seed is different between NVT equil (42), NPT equil (42),
    # and production (99) to avoid any correlations between phases. Using the same seed
    # in all phases would make the random noise sequences correlated — not physically wrong
    # but not best practice. The seed does NOT affect equilibrium properties, only the
    # specific path through phase space. Different seeds give identical thermodynamic
    # averages (within statistical error) for converged simulations.
    integrator_nvt.setRandomNumberSeed(42)

    sim_nvt = app.Simulation(pdb.topology, system_nvt, integrator_nvt, platform)

    nvt_log = f"A10_{label}_nvt.log"
    nvt_dcd = f"A10_{label}_nvt.dcd"
    nvt_chk = f"A10_{label}_nvt.chk"

    # Delete any stale .tmp files left by a previous crashed checkpoint write
    for tmp_f in [nvt_chk + ".0.tmp", nvt_chk + ".1.tmp"]:
        if os.path.exists(tmp_f):
            os.remove(tmp_f)
            print(f"  Removed stale temp file: {tmp_f}")

    # Resume from checkpoint if one exists and NVT is not already complete
    nvt_resume = False
    steps_done = 0
    if os.path.exists(nvt_chk) and os.path.getsize(nvt_chk) > 0:
        sim_nvt.loadCheckpoint(nvt_chk)
        steps_done = sim_nvt.currentStep
        steps_remaining = NVT_STEPS - steps_done
        if steps_remaining <= 0:
            print(f"  NVT already complete ({steps_done:,} steps in checkpoint). Skipping.")
            nvt_resume = True
            nvt_end_state = sim_nvt.context.getState(
                getEnergy=True, getPositions=True, getVelocities=True
            )
        else:
            print(f"  Resuming NVT from checkpoint at step {steps_done:,} "
                  f"({steps_done / STEPS_PER_PS / 1000:.3f} ns). "
                  f"Remaining: {steps_remaining:,} steps "
                  f"({steps_remaining / STEPS_PER_PS / 1000:.3f} ns).")
            nvt_resume = True
    else:
        # Fresh start: seed positions from minimized state
        sim_nvt.context.setPositions(state_post.getPositions())
        sim_nvt.context.setVelocitiesToTemperature(TEMPERATURE_K * unit.kelvin, 42)
        steps_remaining = NVT_STEPS
        print(f"  Starting NVT from scratch ({NVT_STEPS:,} steps).")

    if nvt_resume and steps_remaining <= 0:
        # Already done; skip reporter setup and step() call
        pass
    else:
        # Append to existing log/dcd if resuming, else write fresh
        log_mode = "a" if (nvt_resume and os.path.exists(nvt_log)) else "w"
        sim_nvt.reporters.append(app.StateDataReporter(
            nvt_log, LOG_STEPS,
            step=True, time=True, potentialEnergy=True, kineticEnergy=True,
            temperature=True, progress=True, remainingTime=True,
            totalSteps=NVT_STEPS, separator="\t", append=(log_mode == "a"),
        ))
        sim_nvt.reporters.append(app.StateDataReporter(
            sys.stdout, LOG_STEPS * 10,
            step=True, time=True, temperature=True, progress=True,
            remainingTime=True, totalSteps=NVT_STEPS, separator="\t",
        ))
        # Append to existing DCD if resuming
        sim_nvt.reporters.append(app.DCDReporter(
            nvt_dcd, DCD_STEPS,
            append=(nvt_resume and os.path.exists(nvt_dcd)),
        ))
        sim_nvt.reporters.append(app.CheckpointReporter(nvt_chk, CHK_STEPS))

        print(f"  Log  : {nvt_log}  |  Traj : {nvt_dcd}  |  Ckpt : {nvt_chk}")
        print(f"  Running {steps_remaining:,} steps...")

        t0 = time.time()
        sim_nvt.step(steps_remaining)
        t_nvt = time.time() - t0

        nvt_end_state = sim_nvt.context.getState(getEnergy=True, getPositions=True,
                                                  getVelocities=True)

    ke_nvt = nvt_end_state.getKineticEnergy().value_in_unit(unit.kilojoules_per_mole)
    T_nvt  = 2 * ke_nvt / (DOF * kB)

    if nvt_resume and steps_remaining <= 0:
        print(f"\n  NVT skipped (checkpoint was already complete).")
    else:
        print(f"\n  NVT complete in {t_nvt/60:.1f} min")
    print(f"  Final instantaneous T : {T_nvt:.1f} K  (target: {TEMPERATURE_K} K)")
    if abs(T_nvt - TEMPERATURE_K) > 20:
        print("  WARNING: T deviates >20 K -- consider extending NVT or checking thermostat.")

    # ─────────────────────────────────────────────────────────────────────
    # STEP 3 — NPT EQUILIBRATION (no restraints; fresh system + barostat)
    # ─────────────────────────────────────────────────────────────────────
    section(f"[{label}] Step 3: NPT — {NPT_NS} ns at {TEMPERATURE_K} K / {PRESSURE_BAR} bar")
    print("  Deserializing fresh system for NPT (no positional restraints).")
    print("  Adding Monte Carlo Barostat (volume moves every 25 steps).\n")

    # CORRECTED: always deserialize a fresh system for NPT.
    # DO NOT reuse system_nvt (which has the restraint force attached).
    # Reusing would either retain unwanted restraints or, if you tried to
    # remove them via reinitialize(), risk context state corruption.
    # Fresh deserialization is mandatory here. system_nvt has an extra
    # CustomExternalForce (the positional restraints). If we reused it for NPT,
    # those restraints would remain active, artificially pinning the peptide.
    # XmlSerializer round-trips the System exactly: same forces, parameters,
    # atom order — but as a new Python object with no shared memory with system_nvt.
    with open(xml_path) as f:
        system_npt = XmlSerializer.deserialize(f.read())

    # BAROSTAT CHOICE: MonteCarloBarostat
    # ------------------------------------
    # MonteCarloBarostat performs isotropic volume moves every N=25 steps.
    # At each attempt, the box is scaled by a random factor and the energy
    # difference evaluated; moves accepted by Metropolis criterion. This
    # gives the correct NPT ensemble without modifying equations of motion.
    #
    # WHY NOT Parrinello-Rahman (GROMACS-style):
    # Parrinello-Rahman is a Hamiltonian barostat that modifies the equations
    # of motion to include box vector degrees of freedom. It is more efficient
    # for large systems but requires careful tuning of the piston mass. OpenMM
    # does not implement it. MonteCarloBarostat is the correct OpenMM choice.
    #
    # WHY N=25 (volume move every 25 steps):
    # This is the OpenMM default and the CHARMM-GUI OpenMM export default.
    # Every 25 steps = every 50 fs. This is frequent enough to respond to
    # density fluctuations (timescale ~ps) but not so frequent as to waste
    # compute time on rejected moves. Acceptance rates should be 20-50%;
    # if much lower, the max volume displacement should be reduced (OpenMM
    # adjusts this automatically during equilibration).
    #
    # WHY NOT isotropic assumption:
    # MonteCarloBarostat is isotropic (same scaling in x,y,z). This is correct
    # for a liquid-phase system where there is no preferred direction. Anisotropic
    # barostats are needed for crystals or membranes — not relevant here.
    system_npt.addForce(mm.MonteCarloBarostat(
        PRESSURE_BAR  * unit.bar,
        TEMPERATURE_K * unit.kelvin,
        25,
    ))

    DOF_npt = compute_dof(system_npt)  # same N and constraints, no barostat DOF change

    integrator_npt = mm.LangevinMiddleIntegrator(
        TEMPERATURE_K * unit.kelvin,
        FRICTION_PS   / unit.picosecond,
        TIMESTEP_FS   * unit.femtoseconds,
    )
    integrator_npt.setRandomNumberSeed(42)

    sim_npt = app.Simulation(pdb.topology, system_npt, integrator_npt, platform)

    npt_log = f"A10_{label}_npt.log"
    npt_dcd = f"A10_{label}_npt.dcd"
    npt_chk = f"A10_{label}_npt.chk"

    # Delete stale .tmp files from previous crashed NPT checkpoint writes
    for tmp_f in [npt_chk + ".0.tmp", npt_chk + ".1.tmp"]:
        if os.path.exists(tmp_f):
            os.remove(tmp_f)
            print(f"  Removed stale temp file: {tmp_f}")

    # Resume NPT from checkpoint if one exists
    npt_resume = False
    npt_steps_done = 0
    npt_steps_remaining = NPT_STEPS
    if os.path.exists(npt_chk) and os.path.getsize(npt_chk) > 0:
        sim_npt.loadCheckpoint(npt_chk)
        npt_steps_done = sim_npt.currentStep
        npt_steps_remaining = NPT_STEPS - npt_steps_done
        npt_resume = True
        if npt_steps_remaining <= 0:
            print(f"  NPT already complete ({npt_steps_done:,} steps in checkpoint). Skipping step().")
        else:
            print(f"  Resuming NPT from checkpoint at step {npt_steps_done:,} "
                  f"({npt_steps_done / STEPS_PER_PS / 1000:.3f} ns). "
                  f"Remaining: {npt_steps_remaining:,} steps.")
    else:
        # Seed from end of NVT
        sim_npt.context.setPositions(nvt_end_state.getPositions())
        sim_npt.context.setVelocities(nvt_end_state.getVelocities())
        sim_npt.context.setPeriodicBoxVectors(*nvt_end_state.getPeriodicBoxVectors())

    npt_log_mode = "a" if (npt_resume and os.path.exists(npt_log)) else "w"
    sim_npt.reporters.append(app.StateDataReporter(
        npt_log, LOG_STEPS,
        step=True, time=True, potentialEnergy=True, kineticEnergy=True,
        temperature=True, density=True, progress=True, remainingTime=True,
        totalSteps=NPT_STEPS, separator="\t", append=(npt_log_mode == "a"),
    ))
    sim_npt.reporters.append(app.StateDataReporter(
        sys.stdout, LOG_STEPS * 10,
        step=True, time=True, temperature=True, density=True,
        progress=True, remainingTime=True, totalSteps=NPT_STEPS, separator="\t",
    ))
    sim_npt.reporters.append(app.DCDReporter(
        npt_dcd, DCD_STEPS,
        append=(npt_resume and os.path.exists(npt_dcd)),
    ))
    sim_npt.reporters.append(app.CheckpointReporter(npt_chk, CHK_STEPS))

    print(f"  Log  : {npt_log}  |  Traj : {npt_dcd}  |  Ckpt : {npt_chk}")
    print(f"  Running {npt_steps_remaining:,} steps...")

    t0 = time.time()
    if npt_steps_remaining > 0:
        sim_npt.step(npt_steps_remaining)
    t_npt = time.time() - t0

    # ─────────────────────────────────────────────────────────────────────
    # CONVERGENCE CHECKS
    # ─────────────────────────────────────────────────────────────────────
    section(f"[{label}] Convergence Checks")

    npt_end = sim_npt.context.getState(getEnergy=True, getPositions=True)
    ke_npt  = npt_end.getKineticEnergy().value_in_unit(unit.kilojoules_per_mole)
    T_final = 2 * ke_npt / (DOF_npt * kB)
    density = compute_density_kgm3(system_npt, npt_end)

    print(f"\n  NPT complete in {t_npt/60:.1f} min")
    print(f"  {'Metric':<38} {'Value':>10}   {'Target':>10}   Status")
    print(f"  {'-'*68}")

    t_ok = abs(T_final - TEMPERATURE_K) < 15
    print(f"  {'Final instantaneous T (K)':<38} {T_final:>10.1f}   {TEMPERATURE_K:>10.1f}   "
          f"{'OK' if t_ok else 'WARN extend NVT/NPT'}")

    d_ok = 970 < density < 1030
    print(f"  {'Final density (kg/m3)':<38} {density:>10.1f}   {'~1000':>10}   "
          f"{'OK' if d_ok else 'WARN extend NPT'}")

    # Mean T over full NPT run (from log file)
    try:
        import csv
        temps = []
        with open(npt_log) as f:
            reader = csv.DictReader(f, delimiter="\t")
            for row in reader:
                t_key = [k for k in row if "Temperature" in k]
                if t_key:
                    temps.append(float(row[t_key[0]]))
        if temps:
            mean_T = np.mean(temps)
            std_T  = np.std(temps)
            mt_ok  = abs(mean_T - TEMPERATURE_K) < 5
            print(f"  {'Mean NPT temperature (K)':<38} {mean_T:>10.1f}   {TEMPERATURE_K:>10.1f}   "
                  f"{'OK' if mt_ok else 'WARN'}")
            print(f"  {'NPT temperature std dev (K)':<38} {std_T:>10.1f}   {'< 15':>10}   "
                  f"{'OK' if std_T < 15 else 'WARN'}")
    except Exception as e:
        print(f"  (Log parse skipped: {e})")

    # Backbone RMSD drift across NPT trajectory
    if MDTRAJ_AVAILABLE:
        try:
            traj      = md.load(npt_dcd, top=pdb_path)
            bb_idx    = traj.topology.select("protein and backbone")
            traj_bb   = traj.atom_slice(bb_idx)
            rmsd      = md.rmsd(traj_bb, traj_bb, 0)
            rmsd_end  = float(rmsd[-1])
            rmsd_max  = float(np.max(rmsd))
            r_ok      = rmsd_end < 0.5
            print(f"  {'Backbone RMSD vs start, final (nm)':<38} {rmsd_end:>10.3f}   {'< 0.5':>10}   "
                  f"{'OK' if r_ok else 'WARN large drift'}")
            print(f"  {'Backbone RMSD max (nm)':<38} {rmsd_max:>10.3f}")
        except Exception as e:
            print(f"  (RMSD check skipped: {e})")
    else:
        print("  (RMSD check skipped: mdtraj not available)")

    # ─────────────────────────────────────────────────────────────────────
    # SAVE FINAL NPT STATE
    # ─────────────────────────────────────────────────────────────────────
    final_pdb = f"A10_{label}_npt_final.pdb"
    final_xml = f"A10_{label}_npt_final.xml"

    # enforcePeriodicBox=True wraps all molecules back into the primary box.
    # We serialize system_npt WITHOUT the barostat here so Phase 4 (NVT)
    # can deserialize a clean system and add nothing extra.
    # The box vectors are captured from the final NPT state and baked into
    # the PDB CRYST1 record so Phase 4 knows the equilibrated box size.
    final_state = sim_npt.context.getState(
        getPositions=True, getVelocities=True, enforcePeriodicBox=True
    )
    with open(final_pdb, "w") as f:
        app.PDBFile.writeFile(pdb.topology, final_state.getPositions(), f)

    # Save system WITHOUT barostat for Phase 4 NVT production.
    # Deserialize fresh (no barostat), serialize again.
    with open(xml_path) as f:
        system_for_prod = XmlSerializer.deserialize(f.read())
    with open(final_xml, "w") as f:
        f.write(XmlSerializer.serialize(system_for_prod))

    print(f"\n  NPT final PDB : {final_pdb}")
    print(f"  NVT prod XML  : {final_xml}  (no barostat - for Phase 4)")

    # ─────────────────────────────────────────────────────────────────────
    # RE-SOLVATION FOR EXTENDED CONFORMATION (Kuczera 2021 methodology)
    # ─────────────────────────────────────────────────────────────────────
    # The extended A10 box is much larger than the helix box (the extended
    # peptide is ~3.5 nm end-to-end; its solvated box has ~4x more waters).
    # After NPT equilibration the peptide may have partially collapsed, but
    # the box remains oversized. Re-solvating with a smaller padding (1.2 nm
    # instead of 1.5 nm) produces a box comparable to the helix box, making
    # both production runs cost the same and the results directly comparable.
    # Reference: Kuczera et al. Life 2021, Methods section.
    if label == "extended":
        section(f"[{label}] Re-solvation — compact box for production")
        print("  Extracting equilibrated peptide -> re-solvating in smaller box.")
        print("  This matches the helix box size and follows Kuczera 2021.\n")

        # ── Extract peptide-only topology and positions ───────────────────
        # CRITICAL: we must NOT reconstruct the peptide topology from pdb.topology
        # (the re-read solvated PDB), because app.PDBFile() normalises atom names:
        #   HT1→H, HT2→H2, HT3→H3, HNT→H
        # These renamed atoms then fail forcefield_rs.createSystem() with
        # "No template found" because CHARMM36m's CT3 patch expects HT1/HT2/HT3.
        #
        # Solution: use peptide_top_orig — the in-memory topology returned by
        # build_a10(), which was never passed through app.PDBFile() and retains
        # the correct CHARMM patch atom names.
        #
        # For positions: take the first N_pep positions from the NPT final state.
        # The solvated topology has peptide atoms first (they were added to modeller
        # before addSolvent), so the first N_pep positions in final_state correspond
        # to the peptide atoms in the same order as peptide_top_orig.
        n_pep            = peptide_top_orig.getNumAtoms()
        all_positions_npt = final_state.getPositions()
        peptide_positions = [all_positions_npt[i] for i in range(n_pep)]

        peptide_only_pdb = f"A10_{label}_peptide_only.pdb"
        with open(peptide_only_pdb, "w") as f:
            app.PDBFile.writeFile(peptide_top_orig, peptide_positions, f)
        print(f"  Peptide-only PDB : {peptide_only_pdb}  ({n_pep} atoms, correct CHARMM names)")

        forcefield_rs = app.ForceField("charmm36_2024.xml", "charmm36_2024/water.xml")
        modeller_rs   = app.Modeller(peptide_top_orig, peptide_positions)

        # Use 1.2 nm padding (smaller than initial 1.5 nm) to produce a compact box
        modeller_rs.addSolvent(
            forcefield_rs,
            model         = "tip3p",
            padding       = 1.2 * unit.nanometers,
            ionicStrength = 0.15 * unit.molar,
            neutralize    = True,
        )

        rs_solvated_pdb = f"A10_{label}_resolvated.pdb"
        rs_xml          = f"A10_{label}_resolvated.xml"
        with open(rs_solvated_pdb, "w") as f:
            app.PDBFile.writeFile(modeller_rs.topology, modeller_rs.positions, f)
        print(f"  Re-solvated PDB : {rs_solvated_pdb}")

        system_rs = forcefield_rs.createSystem(
            modeller_rs.topology,
            nonbondedMethod     = app.PME,
            nonbondedCutoff     = 1.2 * unit.nanometers,
            switchDistance      = 1.0 * unit.nanometers,
            constraints         = app.HBonds,
            rigidWater          = True,
            ewaldErrorTolerance = 0.0005,
        )

        n_rs = modeller_rs.topology.getNumAtoms()
        n_rs_wat = sum(1 for r in modeller_rs.topology.residues() if r.name == "HOH")
        print(f"  Re-solvated atoms   : {n_rs:,}  ({n_rs_wat:,} waters)")

        with open(rs_xml, "w") as f:
            f.write(XmlSerializer.serialize(system_rs))

        # Short NPT (2 ns) on the re-solvated system to converge the new box density
        print(f"  Running {NPT_RESOLVATE_NS} ns NPT on re-solvated system...")

        system_rs_npt = XmlSerializer.deserialize(open(rs_xml).read())
        system_rs_npt.addForce(mm.MonteCarloBarostat(
            PRESSURE_BAR * unit.bar, TEMPERATURE_K * unit.kelvin, 25
        ))
        integrator_rs = mm.LangevinMiddleIntegrator(
            TEMPERATURE_K * unit.kelvin, FRICTION_PS / unit.picosecond,
            TIMESTEP_FS * unit.femtoseconds
        )
        integrator_rs.setRandomNumberSeed(42)
        sim_rs = app.Simulation(modeller_rs.topology, system_rs_npt, integrator_rs, platform)
        sim_rs.context.setPositions(modeller_rs.positions)
        sim_rs.context.setVelocitiesToTemperature(TEMPERATURE_K * unit.kelvin, 42)

        rs_log = f"A10_{label}_resolvate_npt.log"
        rs_dcd = f"A10_{label}_resolvate_npt.dcd"
        rs_chk = f"A10_{label}_resolvate_npt.chk"

        rs_resume = os.path.exists(rs_chk)
        if rs_resume:
            print(f"  Resuming re-solvation NPT from checkpoint: {rs_chk}")
            sim_rs.loadCheckpoint(rs_chk)
            rs_steps_done = sim_rs.context.getState().getStepCount()
            rs_remaining  = NPT_RESOLVATE_STEPS - rs_steps_done
            if rs_remaining <= 0:
                print("  Re-solvation NPT already complete.")
                rs_remaining = 0
            else:
                print(f"  Steps done: {rs_steps_done:,}  Remaining: {rs_remaining:,}")
        else:
            rs_remaining = NPT_RESOLVATE_STEPS

        if rs_remaining > 0:
            rs_log_mode = "a" if rs_resume else "w"
            sim_rs.reporters.append(app.StateDataReporter(
                rs_log, LOG_STEPS,
                step=True, time=True, temperature=True, density=True,
                progress=True, remainingTime=True, totalSteps=NPT_RESOLVATE_STEPS,
                separator="\t", append=(rs_log_mode == "a"),
            ))
            sim_rs.reporters.append(app.StateDataReporter(
                sys.stdout, LOG_STEPS * 20,
                step=True, time=True, temperature=True, density=True,
                progress=True, remainingTime=True, totalSteps=NPT_RESOLVATE_STEPS, separator="\t",
            ))
            sim_rs.reporters.append(app.DCDReporter(
                rs_dcd, DCD_STEPS, append=(rs_resume and os.path.exists(rs_dcd))
            ))
            sim_rs.reporters.append(app.CheckpointReporter(rs_chk, CHK_STEPS))
            sim_rs.step(rs_remaining)

        rs_end = sim_rs.context.getState(
            getPositions=True, getVelocities=True, enforcePeriodicBox=True
        )
        rs_density = compute_density_kgm3(system_rs_npt, sim_rs.context.getState())
        print(f"  Re-solvated density : {rs_density:.1f} kg/m3 (target: ~990-1010)")

        # Overwrite final_pdb and final_xml with re-solvated results
        final_pdb = f"A10_{label}_npt_final.pdb"
        final_xml = f"A10_{label}_npt_final.xml"
        with open(final_pdb, "w") as f:
            app.PDBFile.writeFile(modeller_rs.topology, rs_end.getPositions(), f)
        with open(rs_xml) as f_in, open(final_xml, "w") as f_out:
            f_out.write(f_in.read())

        print(f"  Re-solvated final PDB : {final_pdb}")
        print(f"  Re-solvated system XML: {final_xml}")

    print(f"  [{label.upper()}] Phase 3 complete.\n")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 4: NVT PRODUCTION RUN — 500 ns per conformation
# ─────────────────────────────────────────────────────────────────────────────

def run_production(conf: dict):
    """
    NVT production run for one conformation.

    Reads the Phase 3 final PDB and XML (system without barostat) and runs
    PROD_US × 1000 ns of NVT dynamics at TEMPERATURE_K (currently 500 ns).
    The volume is fixed at the NPT-equilibrated value, which is more efficient
    than NPT production and avoids volume-move noise in the trajectory.

    Features
    --------
    - Checkpoint/resume: if A10_{label}_prod.chk exists with progress, continues
      from where it left off rather than restarting.
    - DCD frames every 100 ps (configurable via DCD_INTERVAL_PS).
    - Log every 10 ps with T, PE, KE.
    - Periodic box vectors are set from the NPT final PDB's CRYST1 record.
    """
    label    = conf["label"]
    pdb_path = f"A10_{label}_npt_final.pdb"
    xml_path = f"A10_{label}_npt_final.xml"

    section(f"A10 [{label.upper()}] - Phase 4: NVT Production ({PROD_US} us)")
    print(f"  Input PDB : {pdb_path}")
    print(f"  Input XML : {xml_path}")
    print(f"  Duration  : {PROD_US} us  ({PROD_STEPS:,} steps at {TIMESTEP_FS} fs)")
    print(f"  Temp      : {TEMPERATURE_K} K  (NVT -- volume fixed)")
    print(f"  DCD frame : every {DCD_INTERVAL_PS} ps  ({DCD_STEPS:,} steps)")

    for path in [pdb_path, xml_path]:
        if not os.path.exists(path):
            print(f"\n  ERROR: {path} not found. Run Phase 3 first.")
            return

    pdb = app.PDBFile(pdb_path)
    with open(xml_path) as f:
        system_prod = XmlSerializer.deserialize(f.read())

    N   = system_prod.getNumParticles()
    DOF = compute_dof(system_prod)
    kB  = 8.314e-3

    platform = get_best_platform()
    # PRODUCTION ENSEMBLE: NVT (not NPT)
    # ------------------------------------
    # Once density is converged from NPT equilibration, we switch to NVT
    # (constant Number, Volume, Temperature) for production. Reasons:
    # 1. ACCURACY: NPT volume fluctuations add noise to potential energy and
    #    structural observables. For a small peptide (~4000 atoms), volume
    #    fluctuations are ~0.1% RMS, which is small but adds unnecessary noise
    #    to helix content calculations.
    # 2. SPEED: MonteCarloBarostat overhead is ~5-10% per 25 steps — not large
    #    but there is no benefit to paying it in production.
    # 3. CORRECTNESS: The helix-coil free energy ΔG is a Helmholtz (NVT) quantity.
    #    Switching to NPT would give Gibbs ΔG, which differs by PΔV ≈ 0.001 kcal/mol
    #    — negligible, but NVT is the more natural ensemble for this observable.
    # The box volume is fixed at the NPT-equilibrated value (saved in final_pdb
    # CRYST1 record and loaded below), so we are simulating at the correct density.
    # Reference: Kuczera et al. Life 2021 uses NVT production throughout.
    integrator_prod = mm.LangevinMiddleIntegrator(
        TEMPERATURE_K * unit.kelvin,
        FRICTION_PS   / unit.picosecond,
        TIMESTEP_FS   * unit.femtoseconds,
    )
    integrator_prod.setRandomNumberSeed(99)  # different seed from equilibration

    sim_prod = app.Simulation(pdb.topology, system_prod, integrator_prod, platform)

    prod_log = f"A10_{label}_prod.log"
    prod_dcd = f"A10_{label}_prod.dcd"
    prod_chk = f"A10_{label}_prod.chk"

    # Remove stale tmp checkpoint files
    for tmp_f in [prod_chk + ".0.tmp", prod_chk + ".1.tmp"]:
        if os.path.exists(tmp_f):
            os.remove(tmp_f)

    # Checkpoint/resume logic
    prod_resume   = False
    steps_done    = 0
    steps_remaining = PROD_STEPS
    if os.path.exists(prod_chk) and os.path.getsize(prod_chk) > 0:
        sim_prod.loadCheckpoint(prod_chk)
        steps_done = sim_prod.currentStep
        steps_remaining = PROD_STEPS - steps_done
        prod_resume = True
        if steps_remaining <= 0:
            print(f"\n  Production already complete ({steps_done:,} steps). Skipping.")
            return
        print(f"\n  Resuming from checkpoint: step {steps_done:,} "
              f"({steps_done / STEPS_PER_PS / 1e6:.4f} us). "
              f"Remaining: {steps_remaining:,} steps "
              f"({steps_remaining / STEPS_PER_PS / 1e6:.4f} us).")
    else:
        # Seed from Phase 3 NPT final state
        sim_prod.context.setPositions(pdb.positions)
        sim_prod.context.setVelocitiesToTemperature(TEMPERATURE_K * unit.kelvin, 99)
        # Set box from Phase 3 NPT final PDB (CRYST1 record) — assert non-None
        # so a missing CRYST1 raises immediately with a clear message rather
        # than causing a cryptic PME error later in the run.
        _box = pdb.topology.getPeriodicBoxVectors()
        if _box is None:
            raise ValueError(
                f"No periodic box vectors found in {pdb_path}. "
                "Re-run Phase 3 to regenerate the NPT final PDB with a CRYST1 record."
            )
        sim_prod.context.setPeriodicBoxVectors(*_box)
        print(f"\n  Starting production from scratch ({PROD_STEPS:,} steps).")

    # ─────────────────────────────────────────────────────────────────────
    # WRITE TOPOLOGY PDB BEFORE STARTING THE DCD REPORTER
    # ─────────────────────────────────────────────────────────────────────
    # CRITICAL: MDTraj loads DCD files against a topology file. The topology
    # PDB must reflect the EXACT atom ordering of sim_prod.topology at the
    # moment the DCDReporter starts writing. If this file is missing or
    # uses a different topology (e.g. the pre-resolvation solvated PDB),
    # MDTraj will silently compute wrong torsion angles. Only this file —
    # written here from the live sim_prod context — is safe to use as the
    # MDTraj topology for the production DCD.
    prod_top_pdb = f"A10_{label}_prod_top.pdb"
    if not prod_resume:
        # Write topology only on a fresh start; on resume the existing file
        # already matches the DCD (same sim_prod.topology, same atom order).
        with open(prod_top_pdb, "w") as _f:
            app.PDBFile.writeFile(
                sim_prod.topology,
                sim_prod.context.getState(getPositions=True).getPositions(),
                _f,
            )
        print(f"  Topology PDB : {prod_top_pdb}  (use this — and ONLY this — as MDTraj topology)")

    log_mode = "a" if (prod_resume and os.path.exists(prod_log)) else "w"
    sim_prod.reporters.append(app.StateDataReporter(
        prod_log, LOG_STEPS,
        step=True, time=True, potentialEnergy=True, kineticEnergy=True,
        temperature=True, progress=True, remainingTime=True,
        totalSteps=PROD_STEPS, separator="\t", append=(log_mode == "a"),
    ))
    sim_prod.reporters.append(app.StateDataReporter(
        sys.stdout, LOG_STEPS * 50,   # print to screen every 500 ps
        step=True, time=True, temperature=True, progress=True,
        remainingTime=True, totalSteps=PROD_STEPS, separator="\t",
    ))
    sim_prod.reporters.append(app.DCDReporter(
        prod_dcd, DCD_STEPS,
        append=(prod_resume and os.path.exists(prod_dcd)),
    ))
    sim_prod.reporters.append(app.CheckpointReporter(prod_chk, CHK_STEPS))

    print(f"  Log  : {prod_log}")
    print(f"  Traj : {prod_dcd}  (frame every {DCD_INTERVAL_PS} ps)")
    print(f"  Ckpt : {prod_chk}  (every {CHK_INTERVAL_PS} ps)")
    print(f"  Running {steps_remaining:,} steps...\n")

    t0 = time.time()
    sim_prod.step(steps_remaining)
    t_prod = time.time() - t0

    prod_end = sim_prod.context.getState(getEnergy=True)
    ke_prod  = prod_end.getKineticEnergy().value_in_unit(unit.kilojoules_per_mole)
    T_final  = 2 * ke_prod / (DOF * kB)

    ns_per_day = (steps_remaining / STEPS_PER_PS / 1000) / (t_prod / 86400)

    print(f"\n  Production complete in {t_prod/3600:.2f} h  ({ns_per_day:.0f} ns/day)")
    print(f"  Final instantaneous T : {T_final:.1f} K  (target: {TEMPERATURE_K} K)")
    print(f"  [{label.upper()}] Phase 4 complete.\n")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN RUNNER — Phases 3 + 4 (idempotent; skip completed stages)
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # ── Phase 3: Equilibration ────────────────────────────────────────────
    all_phase3_done = all(
        os.path.exists(f"A10_{c['label']}_npt_final.pdb") and
        os.path.exists(f"A10_{c['label']}_npt_final.xml")
        for c in CONFORMATIONS
    )

    if all_phase3_done:
        section("A10 — PHASE 3: EQUILIBRATION (skipped; cached results found)")
        print("\n  All Phase 3 final PDB/XML outputs already exist on disk.")
        print("  Delete A10_{label}_npt_final.pdb / .xml to force re-run.\n")
    else:
        section("A10 — PHASE 3: EQUILIBRATION")
        print(f"\n  Timestep    : {TIMESTEP_FS} fs")
        print(f"  Temperature : {TEMPERATURE_K} K")
        print(f"  Pressure    : {PRESSURE_BAR} bar")
        print(f"  NVT         : {NVT_NS} ns  (restrained)")
        print(f"  NPT         : {NPT_NS} ns  (unrestrained, barostat on)")
        print(f"  Re-solvate  : extended conformation -> {NPT_RESOLVATE_NS} ns NPT after re-solvation")
        print(f"  Restraint k : {RESTRAINT_K} kJ/mol/nm2  (minimization + NVT only)")

        wall_t0 = time.time()
        for conf in CONFORMATIONS:
            run_equilibration(conf)

        total_h = (time.time() - wall_t0) / 3600
        section("PHASE 3 COMPLETE")
        print(f"\n  Total wall time : {total_h:.2f} h")

    # ── Phase 4: Production ───────────────────────────────────────────────
    section("A10 — PHASE 4: NVT PRODUCTION RUNS")
    print(f"\n  Duration per conformation : {PROD_US} us  ({PROD_STEPS:,} steps)")
    print(f"  Ensemble                  : NVT at {TEMPERATURE_K} K (volume fixed)")
    print(f"  DCD output                : every {DCD_INTERVAL_PS} ps")
    print(f"  Checkpoint                : every {CHK_INTERVAL_PS} ps (resume-safe)\n")

    wall_t0 = time.time()
    for conf in CONFORMATIONS:
        run_production(conf)

    total_h = (time.time() - wall_t0) / 3600
    section("ALL PHASES COMPLETE")
    print(f"\n  Total Phase 4 wall time : {total_h:.2f} h")
    print("\n  Output trajectories:")
    for conf in CONFORMATIONS:
        print(f"    A10_{conf['label']}_prod.dcd   ({int(PROD_US*1000)} ns NVT at {TEMPERATURE_K} K)")
        print(f"    A10_{conf['label']}_prod.log")

    print("""
  ============================================================
  SESSION PERSISTENCE (RunPod / remote Linux)
  ============================================================
  On a cloud instance (RunPod, Vast.ai, etc.) a disconnected SSH or
  Jupyter session will kill this process.  To keep the run alive:

    Option A — tmux (recommended):
      tmux new -s a10sim
      jupyter nbconvert --to script A10_Prod_Run.ipynb
      python A10_Prod_Run.py
      # detach: Ctrl-B then D  |  reattach: tmux attach -t a10sim

    Option B — nohup:
      jupyter nbconvert --to script A10_Prod_Run.ipynb
      nohup python A10_Prod_Run.py > A10_run.out 2>&1 &
      echo $!  # note the PID to monitor with: tail -f A10_run.out

  Checkpoint files (*.chk every {CHK_INTERVAL_PS} ps) allow safe resume after
  any interruption — simply re-run the cell or script.
  """)

All dependencies loaded.

Applied Windows-safe checkpoint patch.

  PHASE 1 — Build A10 Structures (ACE/NME capped)
  [Extended] → A10_extended.pdb
    Residues : 10  (ALA, ALA, ALA, ALA, ALA, ALA, ALA, ALA, ALA, ALA)
    Atoms    : 112  (all-atom including H)
    φ / ψ    : -120.0° / 140.0°
    Capping  : ✓ ACE patch (CY/OY/CAY on first ALA) + CT3 patch (NT/CAT on last ALA)
               Ac-Ala10-NHMe — neutral termini as required by CHARMM36m

  [Alpha-Helix] → A10_helix.pdb
    Residues : 10  (ALA, ALA, ALA, ALA, ALA, ALA, ALA, ALA, ALA, ALA)
    Atoms    : 112  (all-atom including H)
    φ / ψ    : -57.0° / -47.0°
    Capping  : ✓ ACE patch (CY/OY/CAY on first ALA) + CT3 patch (NT/CAT on last ALA)
               Ac-Ala10-NHMe — neutral termini as required by CHARMM36m

[Phase 1] ✓ Both capped peptide topologies built (Ac-Ala10-NHMe via CHARMM ACE/CT3 patches).

  PHASE 2 — Parameterize & Solvate
  Force field : charmm36_2024.xml (CHARMM36m backbone torsions)
  Water model : charmm